# 基于MobilenetV2的情绪识别

## 一、项目背景


------------------------

**表情识别**(facialexpression recognition, FER)是计算机理解人类情感的一个重要方向，也是人机交互的一个重要方面。表情识别是指从静态照片或视频序列中选择出表情状态，从而确定对人物的情绪与心理变化。20世纪70年代的美国心理学家Ekman和Friesen通过大量实验，定义了人类六种基本表情：快乐，气愤，惊讶，害怕，厌恶和悲伤，除此之外后续的分类任务大多增添了一个中性表情。人脸表情识别（FER）在人机交互和情感计算中有着广泛的研究前景，包括人机交互、情绪分析、智能安全等。




## 二、环境和数据集的准备

--------

### 2.1 数据集介绍

本次实验的使用的数据集为Fer2013，它于2013年国际机器学习会议（ICML）上推出，并成为比较表情识别模型性能的基准之一，同时也作为了2013年Kaggle人脸识别比赛的数据。Fer2013包含28709张训练集图像、3589张公开测试集图像和3589张私有测试集图像，每张图像为4848大小的灰度图片，如下图所示。Fer2013数据集中由有生气(angry)、厌恶(disgust)、恐惧(fear)、开心(happy)、难过(sad)、惊讶(surprise)和中性(neutral)七个类别组成。由于这个数据集大多是通过爬虫在互联网上进行爬取所得，因此存在一定的误差性。

![](https://ai-studio-static-online.cdn.bcebos.com/7372be26a2a94844a8cdb722b661d223a647ea87178644ee8f817e26f8bf8967)

### 2.2 数据集准备

将数据集的压缩包解压到任意文件夹中：

有两个数据集:data/data150581/archive.zip 和 images.zip



In [1]:
%cd /home/aistudio

# !unzip -oq data/data150581/archive.zip -d ./emotic

/home/aistudio


然后使用下一步将数据集整理成以文件名表示类别的形式

In [2]:
%cd /home/aistudio
import os
import cv2
import tqdm

def make_new_dir(root_dir, tag, save_dir):
    img_root = os.path.join(root_dir, tag)
    img_dir_list = os.listdir(img_root)
    if not os.path.isdir(save_dir):
        os.mkdir(save_dir)
    filenames = [f for f in os.listdir(save_dir) if f.endswith('.jpg')]
    print("exist len",len(filenames))
    exist_len = len(filenames)

    print(img_dir_list)
    for img_dir in tqdm.tqdm(img_dir_list):
        img_folder = os.path.join(img_root, img_dir)
        
        img_list = [f for f in os.listdir(img_folder) if f.endswith('.jpg') or f.endswith('.png')]
        name_first = img_dir[:2].upper()
        for index, filename in enumerate(img_list):
            img = cv2.imread(os.path.join(img_folder, filename), -1)
            save_name = name_first+"{0:0>5}.jpg".format(index+exist_len)
            cv2.imwrite(os.path.join(save_dir, save_name), img)


# 随机移动10%数量的图像到val文件夹中
import shutil
import os
import random

def move2newDir(inputFolder, saveFolder):
    filenames = [f for f in os.listdir(inputFolder) if f.endswith('.jpg')]
    print("total len",len(filenames))
    random.shuffle(filenames)
    num_val = int(0.1*len(filenames))
    if not os.path.isdir(saveFolder):
        os.mkdir(saveFolder)
        
    for index, filename in enumerate(filenames):  
        src = os.path.join(inputFolder,filename)
        dst = os.path.join(saveFolder,filename)       
        shutil.move(src, dst)
        if index == num_val:
            break

# # #打开以下注释重新整理数据集文件夹
# !rm -rf train/ val/ test/

# root_dir = r"images"
# save_dir = r"train"
# make_new_dir(root_dir, "", save_dir) # 制作训练集

# root_dir = r"ck48"
# save_dir = r"train"
# make_new_dir(root_dir, "", save_dir) # 制作测试集

# root_dir = r"emotic"
# save_dir = r"train"
# make_new_dir(root_dir, "train", save_dir) # 制作测试集

# inputFolder = r"train"
# saveFolder = r"val"
# move2newDir(inputFolder, saveFolder)

/home/aistudio


In [3]:
# 生成paddle数据集
%cd /home/aistudio

import os
import cv2
import numpy as np
from paddle.io import Dataset
# 生成label

class MyDataset(Dataset):
    """
    步骤一：继承 paddle.io.Dataset 类
    """
    def __init__(self, data_dir, transform=None):
        """
        步骤二：实现 __init__ 函数，初始化数据集，将样本和标签映射到列表中
        """
        super(MyDataset, self).__init__()
        self.data_list = []
        self.image_data_list=[]
        filenames = os.listdir(data_dir)
        for name in filenames:
            if name[:2] == 'SA':
                label = 0 
            elif name[:2] == 'DI':
                label = 1
            elif name[:2] == 'HA':
                label = 2
            elif name[:2] == 'FE':
                label = 3 
            elif name[:2] == 'SU':
                label = 4
            elif name[:2] == 'NE':
                label = 5
            elif name[:2] == 'AN':
                label = 6
            else:
                print(name)
                continue
            image_path = os.path.join(data_dir, name)
            self.data_list.append([image_path, label])
            image = cv2.imread(image_path)
            image = cv2.resize(image, (224, 224))
            self.image_data_list.append(image)
        print(len(self.data_list))
        # 传入定义好的数据处理方法，作为自定义数据集类的一个属性
        self.transform = transform

    def __getitem__(self, index):
        """
        步骤三：实现 __getitem__ 函数，定义指定 index 时如何获取数据，并返回单条数据（样本数据、对应的标签）
        """
        # 根据索引，从列表中取出一个图像
        image_path, label = self.data_list[index]
        # 读取图片
        image = self.image_data_list[index]
        

        # 将图片尺寸缩放道 224x224
        
        # 读入的图像数据格式是[H, W, C]
        # 使用转置操作将其变成[C, H, W]
        

        # # image = cv2.imread(image_path)
        if self.transform is not None:
            image = self.transform(image)
        # # 飞桨训练时内部数据格式默认为float32，将图像数据格式转换为 float32
        # image = image.astype('float32')
        # image = image.transpose(2, 0, 1)
        # # 应用数据处理方法到图像上
        
        image = np.transpose(image, (2,0,1))
        image = image.astype('float32')
        # 将数据范围调整到[-1.0, 1.0]之间
        image = image / 255.
        image = image * 2.0 - 1.0

        # CrossEntropyLoss要求label格式为int，将Label格式转换为 int
        label = int(label)
        # 返回图像和对应标签
        return image, label
    def show_image(self,id):
        image = cv2.imread(self.data_list[id][0])
        plt.imshow(image)
    def set_len(self,lenth):
        self.data_list = self.data_list[:lenth]
    def __len__(self):
        """
        步骤四：实现 __len__ 函数，返回数据集的样本总数
        """
        return len(self.data_list)
print("preparing datasets")
from paddle.vision.transforms import Compose, RandomRotation,Resize,Normalize,RandomHorizontalFlip,ColorJitter
transform = Compose([ColorJitter(0.6, 0.5, 0.5, 0.4),RandomRotation(45),RandomHorizontalFlip(0.5)])
# transform = Normalize(mean=[127.5], std=[127.5], data_format='CHW')

# 打印数据集样本数        
train_dataset = MyDataset('train',transform)
val_dataset = MyDataset('val',transform)
print('train_custom_dataset images: ',len(train_dataset), 'test_custom_dataset images: ',len(val_dataset))


/home/aistudio
preparing datasets
38794
4311
train_custom_dataset images:  38794 test_custom_dataset images:  4311


### 定义训练过程
#### 下面利用paddle内置的mobilenetV2预训练模型，完成神经网络训练过程的定义。

In [4]:
import paddle
from paddle.vision.models import MobileNetV2,mobilenet_v2,resnet50,LeNet,mobilenet_v1

print('飞桨框架内置模型：', paddle.vision.models.__all__)
network =paddle.vision.models.mobilenet_v2(pretrained=True,num_classes=7)
# network = MobileNetV2(num_classes=7)
model = paddle.Model(network)
# model.summary((None, 3, 48, 48))
# model = MobileNetV2(pretrained=True,num_classes=7)

飞桨框架内置模型： ['ResNet', 'resnet18', 'resnet34', 'resnet50', 'resnet101', 'resnet152', 'VGG', 'vgg11', 'vgg13', 'vgg16', 'vgg19', 'MobileNetV1', 'mobilenet_v1', 'MobileNetV2', 'mobilenet_v2', 'LeNet']


W1202 09:48:34.181448   637 device_context.cc:447] Please NOTE: device: 0, GPU Compute Capability: 7.0, Driver API Version: 11.2, Runtime API Version: 10.1
W1202 09:48:34.186298   637 device_context.cc:465] device: 0, cuDNN Version: 7.6.
/opt/conda/envs/python35-paddle120-env/lib/python3.7/site-packages/paddle/fluid/dygraph/layers.py:1441: UserWarning: Skip loading for classifier.1.weight. classifier.1.weight receives a shape [1280, 1000], but the expected shape is [1280, 7].
  warnings.warn(("Skip loading for {}. ".format(key) + str(err)))
/opt/conda/envs/python35-paddle120-env/lib/python3.7/site-packages/paddle/fluid/dygraph/layers.py:1441: UserWarning: Skip loading for classifier.1.bias. classifier.1.bias receives a shape [1000], but the expected shape is [7].
  warnings.warn(("Skip loading for {}. ".format(key) + str(err)))


In [ ]:
#训练
# 指定在 CPU/GPU 上训练
import os
if os.path.exists("checkpoint/mobilenet_v2.pdopt"):  
    print("reload model")
    model.load('checkpoint/mobilenet_v2')
# 指定在 GPU 第 0 号卡上训练
# paddle.device.set_device('cpu')
paddle.device.set_device('gpu:0')
scheduler = paddle.optimizer.lr.CosineAnnealingDecay(learning_rate=0.5, T_max=10, verbose=True)#自动学习率,暂时不用
opt = paddle.optimizer.Momentum(learning_rate=0.0001, momentum=0.9, 
        weight_decay=paddle.regularizer.L2Decay(coeff=.00002),
        parameters=model.parameters())
# opt = paddle.optimizer.Adam(learning_rate=0.0003,parameters=model.parameters())
model.prepare(opt,
              paddle.nn.CrossEntropyLoss(),
              paddle.metric.Accuracy())#设置模型训练方式

class SelfDefineCallback(paddle.callbacks.VisualDL):# 自定义会带哦
    def __init__(self):
        super().__init__(log_dir='./log_Res101_sszq')
    def on_epoch_end(self, epoch, logs=None):
        super().on_epoch_end(epoch, logs)
        model.save('checkpoint/mobilenet_v2')
        os.system("echo 'epoch: {}' >> log.txt".format(epoch))


    def on_epoch_begin(self, epoch, logs=None):
        super().on_epoch_begin(epoch, logs)
callback=SelfDefineCallback()
model.fit(train_dataset,# 训练数据集
          val_dataset,# 评估数据集
          epochs=2000,# 总的训练轮次
          batch_size=32,# 批次计算的样本量大小
          num_workers=4,
          verbose=1,# 日志展示格式
          shuffle=True,# 是否打乱样本集
          callbacks=callback)

reload model
Epoch 0: CosineAnnealingDecay set learning rate to 0.5.
The loss value printed in the log is the current step, and the metric is the average value of previous steps.
Epoch 1/2000


/opt/conda/envs/python35-paddle120-env/lib/python3.7/site-packages/paddle/fluid/layers/utils.py:77: DeprecationWarning: Using or importing the ABCs from 'collections' instead of from 'collections.abc' is deprecated, and in 3.8 it will stop working
  return (isinstance(seq, collections.Sequence) and
/opt/conda/envs/python35-paddle120-env/lib/python3.7/site-packages/paddle/nn/layer/norm.py:653: UserWarning: When training, we now always track global mean and variance.
  "When training, we now always track global mean and variance.")


Eval samples: 4311
Epoch 36/2000
step 1213/1213 [==============================] - loss: 1.1219 - acc: 0.8779 - 100ms/step         
Eval begin...
step 135/135 [==============================] - loss: 1.0482 - acc: 0.7759 - 87ms/step          
Eval samples: 4311
Epoch 37/2000
step 1213/1213 [==============================] - loss: 0.5484 - acc: 0.8800 - 99ms/step          
Eval begin...
step 135/135 [==============================] - loss: 0.8456 - acc: 0.7724 - 89ms/step          
Eval samples: 4311
Epoch 38/2000
step 1213/1213 [==============================] - loss: 0.3872 - acc: 0.8810 - 100ms/step          
Eval begin...
step 135/135 [==============================] - loss: 1.0222 - acc: 0.7708 - 86ms/step          
Eval samples: 4311
Epoch 39/2000
step 1213/1213 [==============================] - loss: 0.4515 - acc: 0.8822 - 100ms/step          
Eval begin...
step 135/135 [==============================] - loss: 1.2684 - acc: 0.7861 - 85ms/step          
Eval samples: 4311
Epoch 4

# 预测与导出看另外一个